<a href="https://colab.research.google.com/github/Marssssss/OneAI/blob/main/OneAI_Competitive_Analysis_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneAI 竞品分析 v2 — 2026年6月更新版

> 更新时间：2026-06-16
> 基于：上一轮 14 维度对比分析 + 勘误修正 + 最新代码实现状态 + 2025-2026 竞品格局变化
> 范围：OneAI 项目全部源码（20 crate） vs 全市场 Coding Agent / AI Agent 框架

---

## 第一部分：查漏补缺成果回顾

上一轮分析识别了 7 个 Critical + 8 个 High + 9 个 Medium 级差距。最近的实现提交（Skill系统/项目指令源/原生Grep/MCP握手/流式tool_use修复/SubAgent包装/环境diff注入）已解决了其中 **8 个差距**。

### ✅ 已解决差距（8/24）

| # | 原差距 | 维度 | 原评级 | 新状态 | 实现方式 |
|---|--------|------|--------|--------|---------|
| 1 | DefaultSubAgentFactory todo!() | 子代理 | Critical ✅ | **已实现** | `SubAgentWrapper` + `DefaultSubAgentFactory::create()` 用 AgentLoop 包装 |
| 4 | GrepTool/GlobTool shell out | 工具 | Critical ✅ | **已修复** | 原生 Rust: `walkdir+regex` (Grep) / `glob` crate (Glob) |
| 5 | 无项目指令读取 | 上下文 | Critical ✅ | **已实现** | `ProjectInstructionsSource`: ONEAI.md/CLAUDE.md/AGENTS.md 三格式兼容 |
| 6 | take_snapshot() todo!() | 上下文 | High → Critical | **已实现** | `EnvironmentDiff` + `EnvironmentSnapshot` + `compute_diff()` |
| 22 | Anthropic streaming tool call args 始终为 `"{}"` | 流式 | High | **已修复** | `tool_call_state: HashMap<String, (String, String)>` + `input_json_delta` 累积 |
| 23 | IncrementalStreamParser 未被流式迭代使用 | 流式 | High | **已修复** | `run_streaming_iteration_async()` 现调用 `parser.process_chunk()` |
| 7 | 无 WebFetch 工具 | 工具 | Medium → High | **已实现** | `WebFetchTool` (reqwest + HTML→Markdown) |
| — | NotebookEdit placeholder | 工具 | High | **已修复** | 完整 JSON 解析 + cell replace/insert/delete 操作 |
| — | Skill 系统缺失 | Skill | Low | **已实现** | 8 coding + 6 research + 5 general skills + `SkillSelector` |
| — | hard_max_iterations 应保留为安全兜底 | Agent Loop | Low | **已修复** | `Option<usize>` 默认 `Some(200)` |

### ⚠️ 部分解决（2/24）

| # | 原差距 | 维度 | 状态 | 说明 |
|---|--------|------|------|------|
| 3 | MCP connect/call todo!() | MCP | **部分实现** | Stdio transport JSON-RPC 握手已实现，但 SSE/StreamableHttp 未实现、响应解析不完整、子进程生命周期管理缺失 |
| 1 | 子Agent真实spawn | 子代理 | **部分实现** | `DefaultSubAgentFactory::create()` 可创建 AgentLoop，但：(1) 使用全量工具集而非 scoped 过滤；(2) 未用 `tokio::spawn` 独立线程；(3) `run_paradigm()` 仍返回格式字符串 |

### ❌ 未解决差距（14/24）

| # | 差距 | 维度 | 原评级 | 当前状态 |
|---|------|------|--------|---------|
| 2 | 无真实沙箱 | 安全 | Critical | `SandboxMode::Enabled` 仅 regex 黑名单 |
| 9 | 范式切换语义空洞 | Agent Loop | High | `run_paradigm()` 返回文字而非切换行为 |
| 10 | 无 git worktree 隔离 | 子代理 | High | 无 |
| 8 | 无 StructuredOutput schema 验证 | 子代理 | High | SubAgentSummary 是自由文本 |
| 11 | 无 Context Epoch 增量刷新 | 上下文 | High | EnvironmentDiff 已实现但未接入 Loop |
| 12 | 无 prompt caching | 上下文 | High | Anthropic provider 不设 cache_control |
| 13 | NotebookEdit placeholder → 已修复 | 工具 | — | ✅ |
| 14 | 无 Responses API | 多Provider | High | 仅 Messages API |
| 15 | 工具描述太简 | 工具 | Medium | WebFetchTool 有较长描述，其余仍 1-2 行 |
| 16 | RecoveryManager 未接线 | 错误恢复 | Medium | 仍独立存在 |
| 17 | ProgressiveCheckpoint list todo!() | 记忆 | Medium | 仍 todo!() |
| 18 | SqliteCheckpoint 未实现 | 记忆 | Medium | 仅 FilePersistence |
| 19 | 仅 CodingPack | 域 | Medium | — |
| 20 | 无 Google/Gemini provider | 多Provider | Medium | — |
| — | 无 WebSearch 工具 | 工具 | High | 仅 WebFetch，无搜索聚合 |
| — | 无 ApplyPatchTool | 工具 | High | 仅单点编辑 |
| — | 无 SandboxBackend trait | 安全 | High | 无真实沙箱后端 |

---

## 第二部分：2025-2026 竞品格局扩展

上一轮仅对标 Claude Code / Codex CLI / OpenCode 三个竞品。本轮扩展到 **15 个竞品**，覆盖 IDE Agent、CLI Agent、开源框架、商业平台四大类别。

### 2.1 竞品全景分类

| 类别 | 竞品 | 语言 | 模式 | 核心差异 |
|------|------|------|------|---------|
| **CLI Agent** | Claude Code | TypeScript | 交互式CLI | Anthropic 原生, MCP 核心, 子Agent |
| **CLI Agent** | Codex CLI | TypeScript | CLI + 沙箱 | codex-1 专用模型, Docker 沙箱 |
| **CLI Agent** | OpenCode | Go(TS重构) | TUI | 多Provider, Context Epoch |
| **CLI Agent** | Aider | Python | CLI | Repo Map(tree-sitter), Architect/Editor 双模型 |
| **IDE Agent** | Cursor | TS(VS Code fork) | IDE Agent | Background Agent, YOLO 模式 |
| **IDE Agent** | Cline | TS(VS Code ext) | IDE Agent | 开源免费, MCP, 浏览器工具 |
| **IDE Agent** | Windsurf | TS(VS Code fork) | IDE Agent | Flows(人机同步), 自有模型 |
| **IDE Agent** | Copilot Agent | TS(VS Code ext) | IDE Agent | GitHub 原生, Issue→PR 自动化 |
| **IDE Agent** | JetBrains AI | Kotlin/Java | IDE Agent | 语言特化模型, 深度索引 |
| **IDE Agent** | Amazon Q | — | IDE + Cloud | 代码升级 Agent, AWS 原生 |
| **Cloud Agent** | Google Jules | — | Async Web | 异步委托, Gemini 2.5 Pro |
| **Cloud Agent** | Devin v2 | — | VM 沙箱 | 多 Agent 协作, Knowledge 系统 |
| **Cloud Agent** | Factory Droids | — | CI/CD 集成 | 确定性生产工程 Agent |
| **开源框架** | OpenHands | Python | Docker 沙箱 | 多 Agent 策略, 浏览器工具 |
| **开源框架** | Augment/Sweep | — | 多 Agent | Plan/Search/Implement/Review 四阶段 |
| **通用框架** | OneAI | **Rust** | CLI TUI + FFI | **跨平台通用框架, DomainPack** |

### 2.2 关键维度横向对比

#### Agent Loop 架构

| 竞品 | 循环类型 | 决策类型 | 终止条件 | 范式切换 |
|------|---------|---------|---------|---------|
| Claude Code | while(hasAction) | text/tool_calls | 自然终止 | EnterPlanMode (工具) |
| Codex CLI | while(hasAction) | shell/apply_patch | 自然终止 | 无 |
| Aider | **双循环**(Architect→Editor) | plan/edit | 自然终止 | Architect/Editor 角色切换 |
| Cursor | while(hasAction) | multi-step | 自然终止 | Agent↔Assist 切换 |
| Devin v2 | 多Agent协作 | 每Agent独立循环 | 自然终止 | 不同Agent类型 |
| OpenHands | ReAct | reason+act+observe | 自然终止 | 多策略可选 |
| Augment/Sweep | 四阶段流水线 | plan/search/implement/review | 阶段完成 | 阶段推进 |
| **OneAI** | while(!complete && iter<max) | **4种**(Direct/Tool/Delegate/SwitchParadigm) | Budget+迭代上限 | **范式切换**(语义空洞) |

**分析**：OneAI 的 4 种决策类型是架构设计最丰富的，但 Delegate 和 SwitchParadigm 的语义实现仍然空洞。Aider 的 Architect/Editor 双模型模式是一个值得借鉴的轻量多 Agent 方案——它不需要真正的子 Agent spawn，只需两次 LLM 调用（一次规划、一次执行），可靠性更高。Augment/Sweep 的四阶段流水线是确定性编排的代表。

#### 工具系统

| 竞品 | 工具数 | 原生搜索 | Web工具 | 批量编辑 | 子Agent工具 | MCP |
|------|--------|---------|---------|---------|------------|-----|
| Claude Code | 13+ | 原生ripgrep | WebSearch+WebFetch | ❌(单点Edit) | Agent tool | ✅核心 |
| Codex CLI | 2(核心) | shell替代 | ❌ | apply_patch ✅ | ❌ | ❌ |
| Aider | ~10 | Repo Map(tree-sitter) | ❌ | apply_patch ✅ | Architect(伪) | ❌ |
| Cline | ~15 | shell+文件 | 浏览器工具 | ❌(单点) | ❌ | ✅ |
| Cursor | ~10 | shell+文件 | web搜索 | ❌ | Background Agent | ✅ |
| **OneAI** | 10+1(新增WebFetch) | **原生Rust**(walkdir+regex+glob) | **WebFetch ✅**(无Search) | ❌ | **SubAgentWrapper** | **Stdio握手部分** |

**分析**：OneAI 在原生搜索实现上已与竞品对齐（不再 shell out）。但仍有 3 个工具缺口：
1. **ApplyPatchTool** — Codex、Aider、OpenCode 都有；批量编辑是编码场景刚需
2. **WebSearchTool** — Claude Code、Cline、Cursor 都有；搜索聚合是"编码+查文档"混合场景的必要能力
3. **子Agent scoped tools** — Claude Code 的 Agent tool 按类型过滤工具集；OneAI 当前给子 Agent 全量工具

#### 沙箱/安全

| 竞品 | 沙箱类型 | 进程级隔离 | 权限模型 | 领域配置 |
|------|---------|-----------|---------|---------|
| Claude Code | macOS Seatbelt + Docker | ✅ | 三级(Read/Edit/Bash) | ❌硬编码 |
| Codex CLI | Docker(强制) | ✅ | 三模式(suggest/auto-edit/full-auto) | ❌硬编码 |
| Devin v2 | VM 全隔离 | ✅最强 | 隐式(沙箱边界) | ❌ |
| OpenHands | Docker | ✅ | Docker 隐式 | ❌ |
| Cline | 无沙箱 | ❌ | 审批门 | ❌ |
| **OneAI** | **regex 黑名单** | ❌ | **三级+PermissionProfile** | **✅ DomainPack** |

**分析**：OneAI 的 PermissionProfile + DomainPack 是架构级差异——所有其他竞品都是硬编码的编码权限。但底层的 regex 黑名单是名义沙箱而非真实隔离。这个"配置最灵活但执行最弱"的矛盾需要解决。

#### 记忆/持久化

| 竞品 | STM | LTM | 压缩 | 跨会话 | 项目指令 |
|------|-----|-----|------|--------|---------|
| Claude Code | 对话历史 | memory文件(.claude/) | /compact+harness | ✅ | CLAUDE.md |
| Devin v2 | 对话 | Knowledge系统 | ❌ | ✅ | 团队文档索引 |
| Amazon Q | 对话 | 企业级索引 | ❌ | ✅ | AWS 最佳实践 |
| Aider | Repo Map | ❌ | 截断 | ❌ | ❌ |
| **OneAI** | **STM(滑动窗口)** | **LTM(HNSW+混合评分)** | **BudgetManager+模板** | **✅跨会话** | **✅ ONEAI.md** |

**分析**：OneAI 在记忆架构上是所有竞品中最完整的（STM+LTM+压缩+跨会话+项目指令）。但实现成熟度不足：
- LTM 的 HNSW 向量存储过于简化
- NoopCompressor 是默认值（无真实压缩）
- 混合评分需要 Embedding 服务但默认无

#### 跨平台

| 竞品 | 桌面 | 移动端 | FFI 绑定 | 嵌入式集成 |
|------|------|--------|---------|-----------|
| Claude Code | macOS/Linux/Windows | ❌ | ❌ | ❌ |
| Codex CLI | macOS/Linux/Windows | ❌ | ❌ | ❌ |
| Cursor | macOS/Linux/Windows | ❌ | ❌ | VS Code |
| Cline | macOS/Linux/Windows | ❌ | ❌ | VS Code |
| **OneAI** | **6平台** | **Android/iOS/HarmonyOS** | **Kotlin/Swift/C++/C#** | **任意应用嵌入** |

**分析**：跨平台是 OneAI 的**绝对壁垒**——没有其他任何竞品具备移动端支持和 FFI 绑定。这意味着 OneAI 可以嵌入到：
- Android 应用（通过 Kotlin UniFFI 绑定）
- iOS 应用（通过 Swift UniFFI 绑定）
- 桌面应用（通过 C++ 绑定）
- Unity 游戏引擎（通过 C# 绑定）
- HarmonyOS 应用（通过 callback bridge）

这是所有 CLI/IDE Agent 都做不到的——它们只能在桌面终端运行。

---

## 第三部分：新兴架构模式分析

### 3.1 2025-2026 年 Agent 架构趋势

| 模式 | 描述 | 采用者 | OneAI 覆盖度 |
|------|------|--------|-------------|
| **层级委托** | 编排器分解→专项子Agent执行 | Claude Code, Augment, Devin v2 | **50%**(SubAgentWrapper 存在但 scoped tools/filtering 不完整) |
| **规划→执行分离** | 先规划再执行, 规划和执行用不同模型 | Aider(Architect/Editor), Jules, Amazon Q | **30%**(SwitchParadigm 语义空洞) |
| **自纠正循环** | 执行→验证→迭代直到正确 | Aider, Cursor, Claude Code | **10%**(RecoveryManager 未接线) |
| **持久项目记忆** | 跨会话结构化记忆 | Claude Code(CLAUDE.md), Devin(Knowledge) | **70%**(LTM+项目指令已实现, 但压缩/注入不够深) |
| **MCP 工具扩展** | 标准化协议连接外部工具 | Claude Code, Cline, Cursor, Windsurf | **40%**(Stdio 握手部分实现) |
| **Repo Map / 结构化上下文** | 压缩结构理解而非暴力文件包含 | Aider(tree-sitter), JetBrains(索引) | **0%**(无 tree-sitter 或结构化索引) |
| **声明式 Agent 配置** | 配置文件定义行为而非代码 | Claude Code(CLAUDE.md), OpenCode(AGENTS.md) | **60%**(DomainPack 已声明式, 但缺 YAML/TOML 格式) |
| **成本模型路由** | 简单任务→廉价模型, 复杂任务→贵模型 | 多竞品在探索 | **0%**(ProviderFactory 但无路由逻辑) |
| **异步/后台 Agent** | 任务分配给后台 Agent, 审阅结果 | Cursor(BgAgent), Jules, Codex(cloud) | **0%**(无异步委托机制) |
| **VM/Docker 进程级沙箱** | 真实进程隔离而非名义沙箱 | Devin(VM), Codex(Docker), OpenHands(Docker) | **0%**(仅 regex 黑名单) |
| **浏览器工具** | Agent 可浏览网页/截图/交互 | Cline, OpenHands | **0%**(无) |

### 3.2 Aider 的 Repo Map 模式 — 特别值得借鉴

Aider 的核心创新是 **Repo Map**：用 tree-sitter 解析整个代码库，生成压缩的结构图（类/函数/交叉引用），约 100-200 行即可覆盖一个 10000+ 行的项目。模型在结构图上推理，需要细节时再通过文件工具 drill down。

**为什么值得借鉴**：
- 解决了 OneAI 当前的 `FileTreeSource` 只做目录列表（无结构信息）的问题
- 比 Claude Code 的 `gitStatus 全量注入` 更精准（只注入结构而非文件列表）
- 与 OneAI 的 `DomainPack 第2层(上下文注入)` 完美契合——可以为不同领域生成不同结构的 Repo Map
- tree-sitter 有 Rust 绑定（`tree-sitter` crate），实现成本不高

**OneAI 实现建议**：
```rust
// 新增 StructuredContextSource — tree-sitter Repo Map
pub struct RepoMapSource {
    project_dir: PathBuf,
    language: Option<String>,  // 检测或配置
}

impl ContextSource for RepoMapSource {
    fn key(&self) -> &str { "repo_map" }
    async fn load(&self) -> Result<String> {
        // 1. 检测项目语言 (Cargo.toml → Rust, package.json → JS, etc.)
        // 2. 用 tree-sitter 解析关键文件 → 提取 类/函数/模块/交叉引用
        // 3. 生成压缩结构图 (< 200 lines)
        // 4. 返回结构化文本
    }
    fn refresh_policy(&self) -> RefreshPolicy { RefreshPolicy::OnChange }
    fn priority(&self) -> u32 { 8 }  // 高优先级但低于项目指令
}
```

### 3.3 异步委托模式 — 2026 年新兴方向

Cursor 的 Background Agent 和 Google Jules 代表了一个新范式：**异步委托**——开发者分配任务给 Agent，Agent 在后台独立工作，完成后生成 diff/PR 供审阅。

**为什么值得关注**：
- 适合长任务（大型重构、测试生成、文档编写）——开发者不需要坐等
- 与 OneAI 的 `WorkflowDag + StateGraph` 天然契合——后台 Agent 可以运行预定义工作流
- 与 OneAI 的 `跨平台 FFI` 结合——移动端可以分配任务、稍后审阅结果

**OneAI 实现建议**：
```rust
// 新增 AsyncTaskRunner — 后台任务委托
pub struct AsyncTaskRunner {
    agent_loop: AgentLoop,
    result_store: Arc<dyn TaskResultStore>,
}

impl AsyncTaskRunner {
    pub async fn submit(&self, task: &str) -> TaskId { ... }
    pub async fn get_result(&self, id: &TaskId) -> Option<TaskResult> { ... }
    pub async fn list_pending(&self) -> Vec<TaskId> { ... }
}
```

### 3.4 成本模型路由 — 实用价值高

多个竞品在探索"简单任务→廉价模型, 复杂任务→贵模型"的路由策略。这对 OneAI 的多 Provider 架构是天然契合的。

**OneAI 实现建议**：
```rust
// 在 ProviderFactory 中添加路由逻辑
pub struct ModelRouter {
    rules: Vec<RouteRule>,
}

pub struct RouteRule {
    pattern: String,     // 匹配条件: "task.contains('quick fix')" 等
    model: String,       // 目标模型: "claude-haiku-4-5" 等
    provider: String,    // 目标 provider: "anthropic" 等
}

// AgentLoop 在每轮迭代前评估路由:
// - 简单问题 (DirectAnswer) → 路由到 Haiku/Ollama-small
// - 复杂推理 (Delegate/SwitchParadigm) → 路由到 Opus/GPT-4
```

---

## 第四部分：差距更新与优先级重排

### 4.1 修正后差距全景（按优先级）

#### P0 — 阻塞核心路径（必须立即解决）

| # | 差距 | 维度 | 影响 | 建议方案 |
|---|------|------|------|---------|
| 1 | **范式切换语义空洞** | Agent Loop | SwitchParadigm 返回文字而非切换行为, Delegate/SwitchParadigm 核心特性仍形同虚设 | 实现 `run_paradigm()` 实际切换 system prompt + 工具过滤 + 决策解析 |
| 2 | **无真实沙箱** | 安全 | SandboxMode.Enabled 仅 regex 黑名单, 低于所有主要竞品 | 实现 SandboxBackend trait: macOS Seatbelt / Linux Docker / Windows AppContainer |
| 3 | **MCP 实现不完整** | MCP | Stdio 握手部分工作, SSE/StreamableHttp 不可用, 响应解析不完整, 子进程生命周期缺失 | 用 `rmcp` crate 完整实现 MCP 协议; 或使用成熟的 JSON-RPC framing |
| 4 | **ApplyPatchTool 缺失** | 工具 | 无法批量修改多个文件, 编码场景刚需 | 实现统一 diff 解析 + 原子应用 |
| 5 | **子Agent scoped tools 不完整** | 子代理 | 子Agent 使用全量工具而非按类型过滤, 违反最小权限原则 | 实现 `create_scoped_tools()` 的实际过滤, 仅注入 `available_tools` 中列出的工具定义 |

#### P1 — 短期应解决

| # | 差距 | 维度 | 影响 | 建议方案 |
|---|------|------|------|---------|
| 6 | **WebSearchTool 缺失** | 工具 | 无搜索聚合能力, "编码+查文档" 混合场景受限 | 实现搜索 API 集成 (Serper/SearXNG) |
| 7 | **Context Epoch 未接入 Loop** | 上下文 | EnvironmentDiff 已实现但未在实际循环中注入增量 | 将 `take_snapshot()` + `compute_diff()` 接入 AgentLoop 每轮迭代 |
| 8 | **NoopCompressor 默认** | 压缩 | 长对话上下文膨胀, 最终超出模型窗口 | 实现 TruncationCompressor fallback + LLM 压缩激活条件 |
| 9 | **无 prompt caching** | 上下文 | 每轮重发 system prompt + context, 浪费 token | Anthropic provider 设置 `cache_control: ephemeral` |
| 10 | **无 Responses API** | 多Provider | 仅 Messages API, 无 agent-oriented 优化 | Anthropic provider 实现 Responses API endpoint |
| 11 | **RecoveryManager 未接线** | 错误恢复 | 6 种恢复策略框架存在但 Loop 未使用 | 接入 AgentLoop: 每轮迭代后检查错误并应用恢复策略 |
| 12 | **无 StructuredOutput schema** | 子代理 | SubAgentSummary 是自由文本, 无结构化验证 | 定义 JSON Schema, 子Agent 返回值强制 schema 验证 |
| 13 | **无 git worktree 隔离** | 子代理 | 并行子Agent 可能修改同一文件冲突 | 实现 `git worktree create` 或 directory-level 隔离 |

#### P2 — 中期改进

| # | 差距 | 维度 | 建议 |
|---|------|------|------|
| 14 | 无 Google/Gemini provider | 多Provider | 实现 Gemini Chat API |
| 15 | 仅 CodingPack | 域 | 实现 ResearchPack, DataAnalysisPack |
| 16 | DomainPack 无 YAML/TOML 格式 | 声明式配置 | 添加文件格式解析 |
| 17 | ProgressiveCheckpoint list todo!() | 记忆 | 实现 MemoryCheckpointBackend::list() |
| 18 | SqliteCheckpoint 未实现 | 记忆 | 实现 rusqlite backend |
| 19 | 无 Repo Map (tree-sitter) | 上下文 | 实现 StructuredContextSource |
| 20 | 无异步委托机制 | Agent | 实现 AsyncTaskRunner |
| 21 | 无成本模型路由 | 多Provider | 实现 ModelRouter |
| 22 | 无浏览器工具 | 工具 | 实现 BrowserTool (Playwright) |
| 23 | 子Agent 未用 tokio::spawn | 子代理 | 独立线程运行 |
| 24 | LTM 嵌入提供者未接线 | 记忆 | 接线 FastEmbed 或 LLM embedding |

---

## 第五部分：战略定位与差异化分析

### 5.1 OneAI 的三个不可替代壁垒

1. **跨平台 FFI** — Android/iOS/HarmonyOS/Kotlin/Swift/C++/C# 绑定。没有其他竞品能嵌入移动应用。这决定了 OneAI 的定位不是"终端里的编码工具", 而是"任何应用里的 Agent 能力"。

2. **DomainPack 五层声明式配置** — 工具集/上下文/权限/范式/压缩五个维度可按领域配置。所有其他竞品都是"编码专用硬编码"。OneAI 可以一行切换领域——这在竞品生态中是唯一的。

3. **WorkflowDag + StateGraph** — 确定性工作流编排。其他竞品完全依赖 LLM 作为编排器。OneAI 可以在 Agent Loop 内部嵌入预定义 DAG 执行——这在需要确定性保证的场景（生产部署、合规审查）是刚需。

### 5.2 与竞品的定位差异

| 竞品 | 定位 | OneAI 差异 |
|------|------|-----------|
| Claude Code | 编码专用 Agent CLI | OneAI 是通用框架 + 领域可插拔 |
| Cursor | IDE 编码 Agent | OneAI 可嵌入任何 IDE/应用(非 VS Code 锁定) |
| Aider | CLI 编码配对 | OneAI 有 DomainPack + Workflow + 跨平台 |
| Devin | 云端全自动 AI 工程师 | OneAI 是本地+嵌入框架(非云端锁定) |
| OpenHands | 开源 Docker Agent | OneAI 是 Rust + FFI(非 Python + Docker 限定) |

**核心定位**：OneAI 不是"更好的 Claude Code", 而是"任何应用都能嵌入的 Agent 能力平台"。类比：Claude Code 是 iMessage, OneAI 是通信协议框架——iMessage 只在 Apple 设备上, 通信协议可以在任何设备上。

### 5.3 SWOT 分析

| | 有利 | 不利 |
|---|------|------|
| **内部** | ✅ DomainPack 架构创新<br>✅ 跨平台 FFI 唯一壁垒<br>✅ Rust 性能/安全基础<br>✅ 记忆系统最完整<br>✅ Workflow 确定性编排 | ⚠️ 核心特性实现不足 (范式切换空洞)<br>⚠️ 沙箱名义而非真实<br>⚠️ MCP 实现不完整<br>⚠️ 子Agent scoped tools 缺失 |
| **外部** | ✅ MCP 成为标准协议 → OneAI 实现后可接入生态<br>✅ 移动端 Agent 市场空白 → OneAI 唯一选手<br>✅ 多 Provider 趋势 → OneAI 天然支持<br>✅ Agent 嵌入需求增长 → FFI 绑定价值放大 | ⚠️ Claude Code/Cursor 用户基数巨大<br>⚠️ 开源竞品 (Cline/Aider/OpenHands) 免费<br>⚠️ IDE 集成竞品有天然分发渠道<br>⚠️ 云端 Agent (Devin/Jules) 有算力优势 |

---

## 第六部分：路线图更新

### Phase A: 核心路径打通（2-3 周）— 解决 P0

| 任务 | 文件 | 工作量 |
|------|------|--------|
| 实现 `run_paradigm()` 语义切换 | agent_loop.rs | 中 |
| 实现 SandboxBackend trait + macOS Seatbelt backend | tool_interfaces.rs (新增 sandbox.rs) | 高 |
| 完成 MCP Stdio 实现 (响应解析 + 生命周期管理) | mcp_real.rs | 中 |
| 实现 ApplyPatchTool | tool_interfaces.rs (新增) | 中 |
| 实现子Agent scoped tools 实际过滤 | sub_agent.rs | 低 |

### Phase B: 上下文与 Provider 增强（2 周）— 解决 P1 部分

| 任务 | 文件 | 工作量 |
|------|------|--------|
| Context Epoch 接入 AgentLoop | agent_loop.rs + context_assembler.rs | 中 |
| 实现 TruncationCompressor fallback | budget.rs (新增) | 中 |
| Anthropic prompt caching | anthropic.rs | 低 |
| WebSearchTool (搜索 API) | tool_interfaces.rs (新增) | 中 |
| RecoveryManager 接入 AgentLoop | agent_loop.rs | 低 |

### Phase C: 子Agent 体系完善（2 周）— 解决 P1 部分

| 任务 | 文件 | 工作量 |
|------|------|--------|
| StructuredOutput schema 验证 | sub_agent.rs | 中 |
| git worktree 隔离 (或 directory-level) | 新增 crate/模块 | 高 |
| 子Agent tokio::spawn 异步运行 | sub_agent.rs | 低 |
| Responses API | anthropic.rs (新增) | 高 |

### Phase D: 差异化深化（4 周）— P2 + 新特性

| 任务 | 工作量 |
|------|--------|
| Repo Map (tree-sitter StructuredContextSource) | 中 |
| 异步委托 (AsyncTaskRunner) | 中 |
| 成本模型路由 (ModelRouter) | 中 |
| ResearchPack 基础版 | 中 |
| DomainPack YAML/TOML 格式 | 低 |
| Google/Gemini provider | 中 |
| BrowserTool (Playwright) | 高 |

---

## 第七部分：关键洞察

### 洞察 1：OneAI 不需要"对齐 Claude Code"

Claude Code 是 Anthropic 的编码专用产品，与 Anthropic 模型深度绑定。OneAI 试图对齐它是一个方向性错误。正确的方向是：

**OneAI = Agent 能力基础设施**，Claude Code = Agent 能力产品。

类比：
- Claude Code ≈ Gmail（特定产品的邮件服务）
- OneAI ≈ SMTP + IMAP 协议（邮件能力的通用基础设施）

基础设施不需要"对齐产品"，需要"让产品能在它上面构建"。

### 洞察 2：MCP 是 OneAI 接入生态的关键桥梁

MCP 已成为 Agent 工具扩展的标准协议。Claude Code、Cline、Cursor、Windsurf 都支持 MCP。当 OneAI 完整实现 MCP 后：

- 任何 MCP Server (filesystem、GitHub、Slack、数据库...) 都可以接入 OneAI
- OneAI 的 DomainPack 可以声明 MCP Server 配置，实现领域级的工具自动注册
- 这比"自己实现所有工具"更高效——让 MCP 生态为 OneAI 提供工具

### 洞察 3：移动端 Agent 是蓝海

所有 CLI/IDE Agent 都只能在桌面运行。OneAI 的 Android/iOS/HarmonyOS FFI 绑定使其可以：
- 嵌入到移动应用中为用户提供 Agent 辅助
- 在智能手表/IoT 设备上运行轻量 Agent
- 为车载系统提供 Agent 交互能力

这是 **0 竞争**的市场空间。

### 洞察 4：Rust 性能优势在 Agent 场景下是真实的

Agent 框架的启动时间、内存占用、并发能力直接影响用户体验：
- Claude Code (Node.js): 启动 ~2s, 内存 ~200MB+
- OneAI (Rust): 启动 ~50ms, 内存 ~20MB
- 在移动端（Android/iOS）上，这个差异更大——Node.js 无法嵌入移动应用

### 洞察 5：确定性工作流是合规/生产场景的刚需

所有竞品都完全依赖 LLM 作为编排器——这意味着它们的行为本质上是概率性的。在合规审查、生产部署、金融交易等场景，概率性行为不可接受。OneAI 的 WorkflowDag + StateGraph 可以：
- 定义确定性的审批工作流（每一步都有明确条件）
- 定义确定性的部署工作流（不允许 LLM "跳过" 任何验证步骤）
- 在 Agent Loop 中嵌入确定性检查点

这是所有竞品都做不到的——它们的 Agent 行为完全受 LLM 概率输出驱动。

---

## 附录 A：2025-2026 竞品定价对照

| 竞品 | 定价模式 | 成本 |
|------|---------|------|
| Claude Code | API token 或 Max 订阅 | $0.05-$1/session 或 $100-$200/mo |
| Codex CLI | OpenAI API token | 变动, 需 API key |
| Cursor | 订阅 + 用量 | Free/$20 Pro/$40 Business |
| Aider | 免费(开源) | 仅 API 费用 |
| Cline | 免费(开源 Apache 2.0) | 仅 API 费用, 本地模型 $0 |
| Windsurf | Free + Pro + Enterprise | 分级 |
| Copilot | 订阅 | $10/$19/$39/mo |
| Amazon Q | Free + Pro | Free/$19/mo |
| Jules | 免费(beta) | 免费 |
| Devin v2 | Enterprise | 企业合同 |
| Factory | Enterprise | 未公开 |
| OpenHands | 免费(开源) | 仅 API 费用 |
| **OneAI** | **待定** | **仅 API 费用 + 本地模型 $0** |

## 附录 B：SWE-bench 竞品表现

| Agent | SWE-bench Verified | 方法 |
|-------|-------------------|------|
| Amazon Q | ~50-60% | Agent + 专用模型 |
| Codex | ~40-50% | codex-1 专用模型 |
| OpenHands | ~40-50% | 开源 ReAct |
| SWE-agent + Claude | ~30-40% | 研究框架 |
| Aider | ~30-40% | Architect/Editor |

**OneAI 目标**：CodingPack 端到端可用后, 应在 SWE-bench Verified 上达到 30%+ (与 Aider 对齐), 然后随 DomainPack 深化逐步提升。

## 附录 C：OneAI 竞品优势对照速查

| 维度 | OneAI 是否领先 | 领先程度 |
|------|---------------|---------|
| 跨平台支持 | ✅ 绝对领先 | 唯一有移动端 + FFI |
| DomainPack 配置 | ✅ 绝对领先 | 唯一支持领域切换 |
| Workflow/DAG | ✅ 绝对领先 | 唯一有确定性编排 |
| 记忆系统 | ✅ 领先 | LTM + 跨会话, 竞品无 |
| 可观测性 | ✅ 领先 | OpenInference trace, 竞品无 |
| 多 Provider | ⚠️ 对齐 | 与 Cline/OpenCode/OpenHands 对齐 |
| 原生搜索 | ⚠️ 对齐 | 与 Claude Code 对齐 |
| 沙箱安全 | ❌ 落后 | regex 黑名单 vs Docker/VM |
| MCP 集成 | ❌ 落后 | 部分 vs Claude Code 完整 |
| 批量编辑 | ❌ 落后 | 无 vs Codex/Aider apply_patch |
| 范式切换 | ❌ 落后 | 空洞 vs Aider Architect/Editor |
| IDE 集成 | ❌ 落后 | TUI vs Cursor/Cline VS Code |
